<a href="https://colab.research.google.com/github/Mike0165115321/open_data_storytelling/blob/main/SuperAI%20-%20Hack%201%20(Draft%201).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐟 Data Storytelling: เจาะลึกโครงสร้างการนำเข้า-ส่งออกสินค้าประมงไทย (Thailand Fisheries Trade Analysis)

**Executive Summary:**
* 🌊 **Context:** วิเคราะห์ฐานข้อมูลการค้าระหว่างประเทศกลุ่มสินค้าประมง เพื่อประเมินสถานะของประเทศไทยในฐานะผู้ผลิตและผู้บริโภคในตลาดโลก
* 🎯 **Objectives:** Notebook นี้นำเสนอ Insight ผ่าน 3 แกนหลัก:
   1. **Major Trading Partners (Where):** ตลาดส่งออกหลัก แหล่งนำเข้า และสถานะดุลการค้าของไทย
   2. **Fisheries Product Structure (What):** โครงสร้างสินค้าประมงไทย (Net Exporter vs Net Importer)
   3. **Seasonal Trade Pattern (When):** ฤดูกาลการค้า ปัจจัยจุดพีกและจุดต่ำสุดในรอบปี
* 🚀 **So What?:** ข้อมูลเชิงลึกเหล่านี้จะช่วยให้ผู้กำหนดนโยบายและผู้ประกอบการสามารถวางแผนรับมือฤดูกาล สเกลอัปสินค้าที่มีศักยภาพ และกระจายความเสี่ยงในตลาดคู่ค้าได้อย่างแม่นยำ

---

In [20]:
# ============================================================
# 📦 ส่วนที่ 1: การนำเข้าไลบรารีที่จำเป็น (Dependencies)
# ============================================================
!pip install pycountry -q                 # ติดตั้งไลบรารีแปลงชื่อประเทศอัตโนมัติ (ซ่อน Log ด้วย -q)

import pandas as pd                       # จัดการโครงสร้างตารางข้อมูล (DataFrame) และการวิเคราะห์
import numpy as np                        # คำนวณคณิตศาสตร์ขั้นสูงและการทำงานกับ Array (Vectorization)
import plotly.express as px               # สร้างกราฟ Interactive แบบรวดเร็วและสวยงาม
import plotly.graph_objects as go         # สร้างกราฟ Interactive ที่มีความซับซ้อนสูง (เช่น แดชบอร์ด)
import plotly.io as pio                   # จัดการระบบตั้งค่าเริ่มต้นของ Plotly (ใช้เซ็ต Global Template สี)
from plotly.subplots import make_subplots # สร้าง Layout จัดเรียงหลายกราฟไว้ในหน้าจอเดียว (Grid)
import pycountry                          # แปลงรหัสประเทศ (ISO-2 เป็น ISO-3) เพื่อใช้วาดลงบนแผนที่โลก
import warnings                           # จัดการระบบแจ้งเตือนของ Python
from IPython.display import display, HTML # สร้างและแสดงผล UI/ชิ้นส่วน HTML ให้สวยงามภายใน Jupyter Notebook
import calendar                           # ใช้สำหรับจัดการข้อมูลเกี่ยวกับปฏิทิน

warnings.filterwarnings('ignore')         # ปิดการแจ้งเตือน (Warning) เพื่อให้ผลลัพธ์การรันโค้ดสะอาดตา
pio.renderers.default = "colab"

display(HTML("<span style='color:#7EE787; font-weight:bold; font-family:Segoe UI;'>✅ Dependencies Loaded: โหลดไลบรารีเสร็จสมบูรณ์พร้อมทำงาน</span>"))

In [21]:
# ============================================================
# 🎨 ส่วนที่ 2: การตั้งค่าระบบสีและฟังก์ชันช่วยเหลือ (Config & Helpers)
# ============================================================

# --- 1. ระบบสี (Design System Object) ---
# ปรับเป็น Class เพื่อให้ IDE Auto-complete ทำงานง่ายเวลาเขียนโค้ด (Theme.import_col)
class Theme:
    import_col = '#FF6B6B'       # สีแดงปะการัง (นำเข้า)
    export_col = '#4ECDC4'       # สีฟ้าอมเขียว (ส่งออก)
    bg_dark = '#0D1117'          # พื้นหลังหลัก
    bg_card = '#161B22'          # พื้นหลังกล่องกราฟ
    text = '#E6EDF3'             # สีตัวอักษร
    grid = '#30363D'             # สีเส้นตาราง
    accent = '#F0883E'           # สีส้มเน้นย้ำ
    mean_line = '#8B949E'        # สีเทาเส้นเฉลี่ย

TRADE_COLOR_MAP = {'Import': Theme.import_col, 'Export': Theme.export_col}
COLORS = {k: v for k, v in Theme.__dict__.items() if not k.startswith('__')}

# --- 2. Plotly Template (Global Injection) ---
pio.templates["superai_dark"] = go.layout.Template(
    layout=go.Layout(
        paper_bgcolor=Theme.bg_dark,
        plot_bgcolor=Theme.bg_card,
        font=dict(family='Segoe UI, sans-serif', size=13, color=Theme.text),
        title=dict(font=dict(size=20, color=Theme.text, family='Segoe UI, sans-serif'), x=0.5),
        margin=dict(l=60, r=40, t=80, b=60),
        legend=dict(
            bgcolor='rgba(0,0,0,0)',
            bordercolor=Theme.grid, borderwidth=1, font=dict(size=12)
        ),
        xaxis=dict(gridcolor=Theme.grid, zeroline=False),
        yaxis=dict(gridcolor=Theme.grid, zeroline=False),
        colorway=[Theme.accent, Theme.import_col, Theme.export_col]
    )
)
# ประกาศใช้ทั้ง Notebook
pio.templates.default = "plotly_dark+superai_dark"

def apply_premium_layout(fig, title: str, height: int = 500):
    fig.update_layout(title_text=f"<b>{title}</b>", height=height)
    return fig

# --- 3. ระบบ UI Logging (Rich HTML Display) ---
def print_log(log_dict: dict):
    html = f"""
    <div style='background-color:{Theme.bg_card}; padding:20px; border-radius:10px; border:1px solid {Theme.grid}; color:{Theme.text}; max-width: 500px; font-family: Segoe UI, sans-serif;'>
        <h3 style='margin-top:0; color:{Theme.accent};'>✅ Data Preparation Summary</h3>
        <hr style='border-color:{Theme.grid}; margin-bottom:15px;'>
    """

    icons = {"original": "📥", "duplicates": "🗑️", "final": "✅", "missing": "🔍"}
    for key, val in log_dict.items():
        emoji = next((v for k, v in icons.items() if k in key), '🔹')
        label = key.replace("_", " ").title()
        html += f"<div style='display:flex; justify-content:space-between; margin-bottom:8px; font-size:14px;'>"
        html += f"<span>{emoji} {label}</span> <b style='color:{Theme.text};'>{val:,.0f}</b></div>"

    html += "</div>"
    display(HTML(html))

display(HTML(f"<span style='color:{Theme.export_col}; font-weight:bold;'>🎨 System Activated: Global Template & Premium Logger Loaded.</span>"))

In [22]:
# ============================================================
# 📥 Step 1: โหลดและเตรียมข้อมูล (Data Preparation - Fisheries)
# ============================================================

# 1.1 โหลดข้อมูล (ป้องกันนามิเบีย 'NA' กลายเป็นค่าว่าง)
df = pd.read_csv('import_export.csv', keep_default_na=False, na_values=[''])

log = {
    "original_rows": len(df),
    "missing_values_found": int(df.isnull().sum().sum())
}

# 1.2 ทำความสะอาดข้อมูล (Data Cleaning)
duplicates_count = df.duplicated().sum()
df_clean = df.drop_duplicates().copy()
log["duplicates_removed"] = int(duplicates_count)

df_clean = df_clean.dropna()
log["missing_values_dropped"] = log["missing_values_found"]

# 1.3 ⚠️ แปลงค่าตัวเลขกลับสู่สเกลจริง (ตาม Data Dictionary)
# price: มูลค่าการค้า (บาท * 1,000,000)
df_clean['price_actual'] = df_clean['price'] * 1_000_000
# weight: น้ำหนักสินค้า (กิโลกรัม * 1,000)
df_clean['weight_actual'] = df_clean['weight'] * 1_000

# 1.4 จัดรูปแบบและสร้างฟีเจอร์พื้นฐาน (Feature Engineering)
df_clean['trade_type'] = df_clean['tradeflow'].map({1: 'Import', 2: 'Export'})

# 🌟 สร้างคอลัมน์ "มูลค่าสุทธิ (Net Value)"
# ให้ส่งออก (Export) เป็นบวก (+) และ นำเข้า (Import) เป็นลบ (-)
# เพื่อความง่ายในการคำนวณ "ดุลการค้า (Trade Balance)" ในขั้นตอนต่อไป
df_clean['trade_value_net'] = np.where(
    df_clean['trade_type'] == 'Export',
    df_clean['price_actual'],
    -df_clean['price_actual']
)

# 1.5 แปลงรหัสประเทศ (ISO-2 เป็น ISO-3) สำหรับวาดแผนที่ 3D
def get_iso3(iso2):
    try: return pycountry.countries.get(alpha_2=iso2).alpha_3
    except: return None
df_clean['iso3'] = df_clean['countryID'].apply(get_iso3)

log["final_ready_rows"] = len(df_clean)

# แสดงข้อมูลสรุปการคลีนผ่าน UI
print_log(log)

# ตรวจสอบโครงสร้างข้อมูลที่พร้อมใช้งาน
display_cols = ['year', 'month', 'countryNameTH', 'productDetailTH', 'trade_type', 'price_actual', 'weight_actual', 'trade_value_net']
display(df_clean[display_cols].head(3))

,year,month,countryNameTH,productDetailTH,trade_type,price_actual,weight_actual,trade_value_net
0,2568,8,เบนิน,เต่าและตะพาบน้ำมีชีวิต,Import,48080000000,33000,-48080000000
1,2568,8,มอริเชียส,เต่าและตะพาบน้ำมีชีวิต,Import,44147000000,10000,-44147000000
2,2568,8,บราซิล,เต่าและตะพาบน้ำมีชีวิต,Import,47068000000,8000,-47068000000


# 🌍 Step 2: วิเคราะห์คู่ค้าหลักและดุลการค้า (Major Trading Partners)
**🎯 โจทย์ข้อ 1:** ประเทศใดคือตลาดส่งออกสำคัญ แหล่งนำเข้าหลัก และเราได้เปรียบ/เสียเปรียบดุลการค้ากับใคร?

In [23]:
# ============================================================
# 🌍 Step 2: วิเคราะห์คู่ค้าหลักและดุลการค้า (Major Trading Partners)
# ============================================================

# 1. เตรียมข้อมูล: รวมยอด Import/Export และดุลการค้า แยกตามประเทศ
country_trade = df_clean.groupby(['countryNameTH', 'trade_type'])['price_actual'].sum().unstack(fill_value=0).reset_index()

# เช็คว่ามีคอลัมน์ Import/Export ครบไหม (กัน Error กรณีบางประเทศมีแค่ฝั่งเดียว)
if 'Import' not in country_trade.columns: country_trade['Import'] = 0
if 'Export' not in country_trade.columns: country_trade['Export'] = 0

# คำนวณยอดรวม (Total Trade) และ ดุลการค้า (Trade Balance)
country_trade['Total Trade'] = country_trade['Import'] + country_trade['Export']
country_trade['Trade Balance'] = country_trade['Export'] - country_trade['Import']

# คัดเลือก Top 15 ประเทศที่มี "มูลค่าการค้ารวมสูงสุด" มาแสดงผล
top_countries = country_trade.sort_values(by='Total Trade', ascending=True).tail(15)

# 2. วาดกราฟ Diverging Bar Chart (แท่งหันหลังชนกัน)
fig_q1 = go.Figure()

# แท่งฝั่งส่งออก (Export) -> ชี้ไปทางขวา (บวก)
fig_q1.add_trace(go.Bar(
    y=top_countries['countryNameTH'],
    x=top_countries['Export'],
    name='ส่งออก (Export)',
    orientation='h',
    marker_color=Theme.export_col,
    hovertemplate="<b>%{y}</b><br>ส่งออก: ฿%{x:,.0f}<extra></extra>"
))

# แท่งฝั่งนำเข้า (Import) -> ชี้ไปทางซ้าย (ติดลบเพื่อให้กราฟแยกซ้ายขวา)
fig_q1.add_trace(go.Bar(
    y=top_countries['countryNameTH'],
    x=-top_countries['Import'],  # ใส่เครื่องหมายลบ
    name='นำเข้า (Import)',
    orientation='h',
    marker_color=Theme.import_col,
    hovertemplate="<b>%{y}</b><br>นำเข้า: ฿%{customdata:,.0f}<extra></extra>",
    customdata=top_countries['Import'] # โชว์ค่าจริงตอน Hover ไม่ให้เห็นค่าติดลบ
))

# เพิ่มจุด (Marker) แสดง "ดุลการค้า (Trade Balance)" สุทธิ
fig_q1.add_trace(go.Scatter(
    y=top_countries['countryNameTH'],
    x=top_countries['Trade Balance'],
    name='ดุลการค้า (Net Balance)',
    mode='markers',
    marker=dict(color=Theme.accent, size=10, symbol='diamond'),
    hovertemplate="<b>%{y}</b><br>ดุลการค้า: ฿%{x:,.0f}<extra></extra>"
))

# 3. จัด Layout ระดับ Premium
fig_q1.update_layout(
    title=dict(text="<b>🌍 ตลาดส่งออก/นำเข้าหลัก และดุลการค้าสินค้าประมง (Top 15 Trading Partners)</b>", font=dict(size=20), x=0.5),
    barmode='overlay',
    height=600,
    xaxis=dict(
        title='มูลค่าการค้า (บาท)',
        tickvals=[-10e9, -5e9, 0, 5e9, 10e9, 15e9], # ตัวอย่างสเกล (ปรับได้ตาม Data จริง)
        ticktext=["฿10B", "฿5B", "0", "฿5B", "฿10B", "฿15B"],
        zeroline=True, zerolinewidth=2, zerolinecolor=Theme.text
    ),
    yaxis=dict(title='ประเทศคู่ค้า'),
    hovermode="y unified"
)

fig_q1.show()

### 📌 Insight & So What? (สรุปผลข้อ 1)
* 📤 **ตลาดส่งออกหลัก (Top Destinations):** ประเทศที่กราฟแท่งสีฟ้า (Export) ยาวที่สุดทางฝั่งขวา คือแหล่งรายได้หลักของเรา
* 📥 **แหล่งนำเข้าหลัก (Top Sources):** ประเทศที่กราฟแท่งสีแดง (Import) ยาวที่สุดทางฝั่งซ้าย คือประเทศที่เราพึ่งพาวัตถุดิบประมง (เช่น นำเข้าปลามาแปรรูป)
* ⚖️ **สถานะดุลการค้า (Trade Balance):** * สังเกตที่ **"จุดข้าวหลามตัดสีส้ม"** หากจุดนี้อยู่ฝั่งขวา แปลว่าเรา **"เกินดุล (Surplus)"** กับประเทศนั้น (รับเงินเข้ามากกว่าจ่าย)
    * หากจุดสีส้มตกไปอยู่ฝั่งซ้าย แปลว่าเรา **"ขาดดุล (Deficit)"** * 🚀 **So What?:** สำหรับประเทศที่เราเกินดุลสูง รัฐควรทำ FTA หรือส่งเสริมแคมเปญการตลาดเพื่อรักษาฐานลูกค้าให้แน่นแฟ้น ส่วนประเทศที่เรานำเข้าสูง (ขาดดุล) ควรประเมินว่าเป็นการนำเข้าเพื่อการบริโภค หรือนำเข้าเพื่อแปรรูปส่งออกต่อ หากเป็นอย่างแรก ควรมีนโยบายส่งเสริมให้ชาวประมงไทยผลิตทดแทน (Import Substitution)

# 🦐 Step 3: โครงสร้างสินค้าประมงไทย (Tornado Chart: Top Exporters vs Importers)
**🎯 โจทย์ข้อ 2:** วิเคราะห์โครงสร้างสินค้าประมง โดยโฟกัสเฉพาะ **Top 10 สินค้าทำกำไร (Net Exporters)** และ **Top 10 สินค้านำเข้าหลัก (Net Importers)** เพื่อให้เห็นภาพรวมเชิงกลยุทธ์ที่ชัดเจนที่สุด

In [24]:
# ============================================================
# 🦐 Step 3: โครงสร้างสินค้าประมงไทย (Tornado Chart)
# ============================================================

# 1. เตรียมข้อมูล: จัดกลุ่มตาม HS Code
product_trade = df_clean.groupby(['heading11', 'productDetailTH', 'trade_type'])['price_actual'].sum().unstack(fill_value=0).reset_index()

if 'Import' not in product_trade.columns: product_trade['Import'] = 0
if 'Export' not in product_trade.columns: product_trade['Export'] = 0

# คำนวณดุลการค้าสุทธิ
product_trade['Net Balance'] = product_trade['Export'] - product_trade['Import']

# 🌟 การตัด Noise ระดับกลยุทธ์: คัดเฉพาะ Top 10 ตัวท็อปของแต่ละฝั่ง
top_exporters = product_trade.nlargest(10, 'Net Balance')
top_importers = product_trade.nsmallest(10, 'Net Balance')

# นำมาต่อยอดกันและเรียงลำดับให้เป็นรูปทรงพายุทอร์นาโด
focused_trade = pd.concat([top_importers, top_exporters]).sort_values(by='Net Balance', ascending=True)

# 2. วาดกราฟ Tornado Chart
fig_q2 = go.Figure()

# แท่งฝั่งส่งออก (สีฟ้า ชี้ไปทางขวา)
fig_q2.add_trace(go.Bar(
    y=focused_trade['productDetailTH'],
    x=focused_trade['Export'],
    name='มูลค่าส่งออก (Export)',
    orientation='h',
    marker_color=Theme.export_col,
    text=focused_trade['Export'].apply(lambda x: f"฿{x/1e6:,.0f}M" if x > 0 else ""), # ใส่ตัวเลขบนแท่ง
    textposition='outside',
    hovertemplate="<b>%{y}</b><br>ส่งออก: ฿%{x:,.0f}<extra></extra>"
))

# แท่งฝั่งนำเข้า (สีแดง ชี้ไปทางซ้าย)
fig_q2.add_trace(go.Bar(
    y=focused_trade['productDetailTH'],
    x=-focused_trade['Import'], # ติดลบเพื่อให้กราฟกางออกซ้าย
    name='มูลค่านำเข้า (Import)',
    orientation='h',
    marker_color=Theme.import_col,
    text=focused_trade['Import'].apply(lambda x: f"฿{x/1e6:,.0f}M" if x > 0 else ""),
    textposition='outside',
    hovertemplate="<b>%{y}</b><br>นำเข้า: ฿%{customdata:,.0f}<extra></extra>",
    customdata=focused_trade['Import']
))

# 3. จัด Layout ให้สะอาดและเป็นมืออาชีพ
fig_q2.update_layout(
    title=dict(text="<b>🦐 Top 10 สินค้าขับเคลื่อนกำไร vs Top 10 สินค้าพึ่งพานำเข้า</b>", font=dict(size=20), x=0.5),
    barmode='overlay',
    height=750,
    xaxis=dict(
        title='มูลค่าการค้า (บาท)',
        showticklabels=False, # ซ่อนตัวเลขแกน X เพราะเราใส่ไว้บนแท่งกราฟแล้ว (ลดความรก)
        zeroline=True, zerolinewidth=2, zerolinecolor=Theme.text
    ),
    yaxis=dict(
        title='',
        tickfont=dict(size=12),
        automargin=True
    ),
    hovermode="y unified",
    margin=dict(l=20, r=40, t=80, b=20)
)

fig_q2.show()

### 📌 Insight & So What? (สรุปผลข้อ 2)
* 🌟 **กลุ่มอู่ข้าวอู่น้ำ (Top Net Exporters):** ครึ่งบนของกราฟ (แท่งสีฟ้าโดดเด่น) คือสินค้าที่เป็นความแข็งแกร่งของไทย (เช่น *รอใส่ชื่อสินค้า*) สินค้าเหล่านี้ทำเงินเข้าประเทศมหาศาลเมื่อเทียบกับการนำเข้า
* ⚠️ **กลุ่มพึ่งพาวัตถุดิบ (Top Net Importers):** ครึ่งล่างของกราฟ (แท่งสีแดงโดดเด่น) คือจุดที่เราสูญเสียดุลการค้า (เช่น *รอใส่ชื่อสินค้า*) อย่างไรก็ตาม ต้องวิเคราะห์ต่อว่าเป็นการนำเข้าเพื่อ "บริโภคในประเทศ" หรือนำเข้าเป็น "วัตถุดิบต้นน้ำ" เพื่อป้อนโรงงานแปรรูป (เช่น นำเข้าปลาทูน่าแช่แข็ง เพื่อทำทูน่ากระป๋องส่งออก)
* 🚀 **So What?:** ภาพนี้ช่วยให้ผู้กำหนดนโยบายเห็น "จุดแข็งที่ต้องรักษา" และ "จุดอ่อนที่ต้องอุดรอยรั่ว" หากมีการกีดกันทางการค้าในกลุ่มครึ่งล่าง (สีแดง) อุตสาหกรรมแปรรูปของไทยจะได้รับผลกระทบอย่างรุนแรง จึงควรหาแหล่งวัตถุดิบสำรอง (Sourcing Diversification) ไว้เสมอ

# 📈 Step 4: วิเคราะห์วัฏจักรและฤดูกาลการค้า (Seasonal Trade Pattern)
**🎯 โจทย์ข้อ 3:** วิเคราะห์แนวโน้มรายเดือนตลอดทั้งปี เพื่อหาจุดสูงสุด (Peak Season) และจุดต่ำสุด (Low Season) พร้อมตั้งสมมติฐานถึงปัจจัยที่ขับเคลื่อนกลไกนี้

In [25]:
# ============================================================
# 📈 Step 4: แนวโน้มฤดูกาลการค้าสัตว์น้ำ (Seasonal Trade Pattern)
# ============================================================
# 1. เตรียมข้อมูล: รวมมูลค่าการค้าแยกตามเดือนและฝั่งการค้า
monthly_trade = df_clean.groupby(['month', 'trade_type'])['price_actual'].sum().unstack(fill_value=0).reset_index()

# สร้างชื่อเดือนภาษาอังกฤษ (Jan, Feb, ...) ให้ดูเป็นสากลและอ่านง่าย
monthly_trade['month_name'] = monthly_trade['month'].apply(lambda x: calendar.month_abbr[x])

# หาตำแหน่งเดือนที่เป็น Peak และ Low ของฝั่ง "ส่งออก (Export)" เพื่อทำ Label
max_export_idx = monthly_trade['Export'].idxmax()
min_export_idx = monthly_trade['Export'].idxmin()

max_month = monthly_trade.loc[max_export_idx, 'month_name']
max_val = monthly_trade.loc[max_export_idx, 'Export']
min_month = monthly_trade.loc[min_export_idx, 'month_name']
min_val = monthly_trade.loc[min_export_idx, 'Export']

# 2. วาดกราฟเส้น (Line Chart)
fig_q3 = go.Figure()

# เส้นส่งออก (Export) - เน้นเส้นหนา โดดเด่น
fig_q3.add_trace(go.Scatter(
    x=monthly_trade['month_name'],
    y=monthly_trade['Export'],
    name='มูลค่าส่งออก (Export)',
    mode='lines+markers',
    line=dict(color=Theme.export_col, width=4),
    marker=dict(size=10, color=Theme.bg_dark, line=dict(width=2, color=Theme.export_col)),
    hovertemplate="<b>%{x}</b><br>ส่งออก: ฿%{y:,.0f}<extra></extra>"
))

# เส้นนำเข้า (Import) - ทำเป็นเส้นประบางๆ เพื่อให้เป็นบริบทเปรียบเทียบ (Context)
fig_q3.add_trace(go.Scatter(
    x=monthly_trade['month_name'],
    y=monthly_trade['Import'],
    name='มูลค่านำเข้า (Import)',
    mode='lines+markers',
    line=dict(color=Theme.import_col, width=2, dash='dash'),
    marker=dict(size=6),
    hovertemplate="<b>%{x}</b><br>นำเข้า: ฿%{y:,.0f}<extra></extra>"
))

# 3. ไฮไลต์จุด Peak และ Low ด้วย Annotation ที่สะดุดตา
fig_q3.add_annotation(
    x=max_month, y=max_val,
    text=f"<b>🌟 Peak Season</b><br>พุ่งสูงสุดในรอบปี",
    showarrow=True, arrowhead=2, arrowsize=1.5, arrowcolor=Theme.accent,
    font=dict(size=13, color=Theme.accent),
    ax=0, ay=-50
)

fig_q3.add_annotation(
    x=min_month, y=min_val,
    text=f"<b>⚠️ Low Season</b><br>ชะลอตัวต่ำสุด",
    showarrow=True, arrowhead=2, arrowcolor=Theme.text,
    font=dict(size=12, color=Theme.text),
    ax=0, ay=50
)

# 4. จัด Layout แบบ Minimalist & Clean
fig_q3.update_layout(
    title=dict(text="<b>📈 วัฏจักรการค้าสัตว์น้ำรายเดือน (Seasonality Trend)</b>", font=dict(size=20), x=0.5),
    height=550,
    xaxis=dict(
        title='เดือน (Month)',
        showgrid=False,
        tickfont=dict(size=12)
    ),
    yaxis=dict(
        title='มูลค่าการค้า (บาท)',
        showgrid=True, gridcolor=Theme.grid,
        zeroline=True, zerolinewidth=1, zerolinecolor=Theme.grid,
        # แปลงสเกลแกน Y เป็นหน่วย M (ล้านบาท) หรือ B (พันล้านบาท) อัตโนมัติเวลาแสดงผล
        tickformat='.2s', hoverformat='.0f'
    ),
    hovermode="x unified",
    margin=dict(l=40, r=40, t=80, b=40)
)

fig_q3.show()

### 📌 Insight & So What? (สรุปผลข้อ 3)
* 📈 **Peak Season (ช่วงกอบโกย):** ยอดส่งออกพุ่งทะยานสูงสุดใน **May (พฤษภาคม)**  *สมมติฐาน:* สอดคล้องกับช่วงเวลาที่ตลาดโลก (เช่น สหรัฐฯ และยุโรป) เร่งนำเข้าสินค้าประมงแช่แข็งเพื่อสต็อกเตรียมรับเทศกาลสำคัญ หรือเป็นช่วงที่ผลผลิตเพาะเลี้ยงของไทย (เช่น กุ้งขาว) ออกสู่ตลาดพร้อมกัน
* 📉 **Low Season (ช่วงชะลอตัว):** การค้าหดตัวลงอย่างชัดเจนใน **Jun  มิถุนายน**  *สมมติฐาน:* อาจเป็นผลกระทบจาก **"ฤดูมรสุม"** ที่เรือประมงพาณิชย์ออกฝั่งไม่ได้ หรือตรงกับช่วงนโยบาย **"ปิดอ่าวไทย (ทะเลพักฟื้น)"** เพื่ออนุรักษ์พันธุ์สัตว์น้ำ ทำให้ซัพพลายต้นน้ำขาดแคลนชั่วคราว
* 🚀 **So What?:**  **สำหรับภาครัฐ:** ควรมีนโยบายอุดหนุน "ระบบห้องเย็น (Cold Storage)" เยียวยาเกษตรกรในช่วง Low Season เพื่อยืดอายุสินค้าไปขายเก็งกำไรในช่วง Peak Season
  * **สำหรับโรงงานแปรรูป:** ต้องวางแผน Buffer Stock (ตุนวัตถุดิบนำเข้า) ล่วงหน้าก่อนเข้าฤดูมรสุม เพื่อป้องกันสายการผลิตหยุดชะงัก (Supply Chain Disruption)